## Computational Astrobiology – Project Dataset Harvesting

---

This script is designed to process simulated exoplanetary system data generated by the previous code. It handles the post-processing phase of an exoplanet detection pipeline, ingesting the raw simulation metadata (ground truths) and extracting parsed target features to model theoretical astrobiological metrics, evaluate light curve variability, analyze transit parameters discovered via **Box Least Squares (BLS)**, and mathematically invert observable properties to determine physical planetary parameters.

---

#### 1. Data Architecture & Naming Conventions

To keep tracking clean across complex evaluation steps, the script uses a strict variable prefix naming convention:

| Prefix | Category | Source / Description |
| --- | --- | --- |
| `t_` | **Ground Truth** | Directly parsed from the underlying simulation YAML configuration files. |
| `f_` | **Feature Value** | Extracted from target text headers or calculated directly from raw empirical observations. |
| `p_` | **Predicted Metric** | Evaluated via theoretical models using the baseline ground truth context. |
| `c_` | **Calculated Value** | Mathematically derived by inverting the observed data parameters. |

##### Parse Mechanics & Stellar Bulk Density

The script uses regular expressions to isolate core stellar values from the internal file log (`group['txt']`), locating lines detailing system attributes like mass, temperature, radius, gravity, and luminosity. Once isolated, it calculates the **Stellar Bulk Density Feature ($f_{\text{dens}}$)**:

$$f_{\text{dens}} = \frac{f_{\text{mass}}}{f_{\text{radius}}^3}$$

This parameter calculates density in solar units ($\rho / \rho_\odot$). It acts as an anchor for vetting the physical reality of any subsequent transit detections.

#### 2. Theoretical Astrobiological & Activity Physics

The pipeline implements an analytical model to calculate the overall habitability and survival probability of the system under stellar magnetic activity.

##### Stellar Activity Hostility ($p_H$)

The code models how much starspots and flares impact the planetary environment using an exponential complement-of-survival model. First, it computes baseline activity scaling factors:

$$\tau_{\text{term}} = \frac{t_{\text{act\_sigma}}}{t_{\text{act\_tau}}}$$

$$\text{flare}_{\text{term}} = \frac{t_{\text{act\_flare\_amp}} \times t_{\text{act\_flare\_dur}}}{t_{\text{act\_flare\_per}}}$$

These terms calculate the energy dissipation rates of magnetic phenomena. Using these rates, it derives individual impact probabilities for background activity ($p_B$) and flares ($p_F$) alongside an integrated **Stellar Hostility Score ($p_H$)**:

$$p_B = 1.0 - \exp\left(-k_\sigma \cdot \tau_{\text{term}}\right)$$

$$p_F = 1.0 - \exp\left(-k_{\text{flare}} \cdot t_{\text{act\_flare}} \cdot \text{flare}_{\text{term}}\right)$$

$$p_H = 1.0 - (1.0 - p_B)(1.0 - p_F)$$

Where $k_\sigma = 0.0010$ and $k_{\text{flare}} = 0.0025$ act as vulnerability coefficients. The final value $p_H \in [0, 1]$ represents the probability that stellar activity severely degrades planetary habitability.

##### Radial Match Penalty ($p_R$)

To score how close the planet is to an Earth-sized analog, the code uses a flat-topped, higher-order exponential penalty kernel:

$$p_R = \exp\left(-15 \times (t_{\text{planet\_r}} - 1.0)^4\right)$$

This function equals $1.0$ when $t_{\text{planet\_r}} = 1.0\ R_\oplus$, but drops off sharply for larger super-Earths or smaller sub-Earths.

##### Habitable Zone Flux Bounds ($p_T$)

To find the insolation limits of the Habitable Zone (HZ), the pipeline implements a 4th-order polynomial expansion based on Kopparapu et al. models. Using the stellar effective temperature deviation $ts = f_{\text{teff}} - T_{\text{eff},\odot}$:

$$\text{HZ}_{\text{inner}} = 1.1074 + 1.3323\times 10^{-4}\cdot ts + 1.5804\times 10^{-8}\cdot ts^2 - 8.3081\times 10^{-12}\cdot ts^3 - 1.9314\times 10^{-15}\cdot ts^4$$

$$\text{HZ}_{\text{outer}} = 0.3560 + 6.1713\times 10^{-5}\cdot ts + 1.6981\times 10^{-9}\cdot ts^2 - 3.1982\times 10^{-12}\cdot ts^3 - 5.5753\times 10^{-16}\cdot ts^4$$

The planetary insolation flux ($S$) and HZ center-matching statistics are computed via:

$$S = \left(\frac{f_{\text{radius}}}{t_{\text{planet\_a}}}\right)^2 \times \left(\frac{f_{\text{teff}}}{T_{\text{eff},\odot}}\right)^4$$

$$S_{\text{mid}} = \frac{\text{HZ}_{\text{inner}} + \text{HZ}_{\text{outer}}}{2.0}, \quad S_{\text{half}} = \frac{\text{HZ}_{\text{inner}} - \text{HZ}_{\text{outer}}}{2.0}$$

$$x_{\text{eff}} = \frac{S - S_{\text{mid}}}{S_{\text{half}}} \implies p_T = \exp\left(-x_{\text{eff}}^6\right)$$

> **Mathematical Method:** Raising $x_{\text{eff}}$ to the 6th power creates a super-Gaussian boxcar filter. The score $p_T$ stays near $1.0$ across the entire width of the habitable zone ($|x_{\text{eff}}| \le 1$) and drops sharply to $0.0$ immediately outside its boundaries.

The total habitability metric ($p_C$) combines these size and temperature scores while penalizing for stellar hostility:

$$p_C = \left(0.5 \cdot p_T + 0.5 \cdot p_R\right) \times (1.0 - p_H)$$

#### 3. Light Curve Statistical Feature Engineering

Before analyzing transits, the script cleans the light curve data and computes descriptive statistical features to characterize instrument noise, stellar oscillations, and residual variability:

* **Dispersion Metrics:** Root Mean Square (`f_lc_rms`) and Median Absolute Deviation (`f_lc_mad`) measure high-frequency noise and residual stellar activity.
* **Percentile Bounds:** Tracks the 1%, 5%, 95%, and 99% flux levels to measure asymmetrical variations caused by deep starspots or energetic flares.
* **Higher-Order Moments:** Skewness measures flux distribution asymmetry (e.g., in-transit dips yield negative skew, flares yield positive skew), while Kurtosis measures the presence of extreme outliers relative to Gaussian white noise.

#### 4. Transit Signals & BLS Geometry Verification

When a periodic signal is detected via the Box Least Squares (BLS) algorithm, the script isolates the top three periodic power peaks ($f_{\text{bls\_period\_1st, 2nd, 3rd}}$) and computes their ratios to identify integer harmonics or aliasing artifacts.

##### Diagnostic Verification Metrics

* **Transit Coverage Ratio:** Computes the ratio of observed transits containing data points to the total expected transits based on baseline duration:

$$f_{\text{bls\_num\_ratio}} = \frac{N_{\text{transits, actual}}}{N_{\text{transits, expected}}}$$


* **Transit Shape Factor:** Measures how much the transit profile matches a classic box shape versus a grazing V-shape by comparing mean and maximum depths:

$$f_{\text{bls\_trans\_shape\_fac}} = \frac{\bar{\delta}}{\delta_{\max}}$$



A value near $1.0$ indicates a flat-bottomed, U-shaped transit with long occultation phases, while a value near $0.5$ signals a grazing, V-shaped transit profile.

##### Stellar Density Back-Calculation

Assuming a circular orbit ($e=0$) with a central impact parameter ($b=0$), the code dynamically reconstructs the host star's density from observable transit features:

$$\rho_{\text{calc}} = \frac{3\pi}{G} \left( \frac{P}{\pi \cdot \Delta t} \right)^3$$

Where $P$ is the orbital period and $\Delta t$ is the total transit duration converted to seconds. The script then computes the verification ratio:

$$f_{\text{density\_ratio}} = \frac{\rho_{\text{calc}}}{\rho_\star}$$

If $f_{\text{density\_ratio}}$ deviates significantly from $1.0$, it indicates that the circular orbit assumption is incorrect, an eccentric orbit ($e > 0$) is present, or the transit is grazing ($b > 0$).

#### 5. Inversion of Planetary Parameters

The pipeline takes the raw transit light curve features (period $P$, duration $\Delta t$, and depth $\delta$) and inverts them back into physical planetary properties.

```
       Observable Transit Features
      [ Period (P), Duration (Δt), Depth (δ) ]
                        │
                        ▼
         ┌──────────────────────────────┐
         │  Keplerian System Inversion  │
         └──────────────┬───────────────┘
                        │
      ┌─────────────────┼─────────────────┐
      ▼                 ▼                 ▼
[ Planet Radius ]  [ Semi-Major Axis ]  [ Inclination ]
  c_planet_r         c_planet_a           c_planet_i

```

##### Calculated Planet Radius ($c_{\text{planet\_r}}$)

Derived from the flux drop ratio, converting solar radius values back into Earth units:

$$c_{\text{planet\_r}} = f_{\text{radius}} \times \sqrt{f_{\text{best\_depth}}} \times 109.12 \quad \left[\text{where } \frac{R_\odot}{R_\oplus} \approx 109.12\right]$$

##### Calculated Semi-Major Axis ($c_{\text{planet\_a}}$)

Derived by solving Kepler's Third Law for a planet with negligible mass ($M_p \ll M_\star$), using standardized solar-system units:

$$c_{\text{planet\_a}} = \left( f_{\text{mass}} \times \left(\frac{f_{\text{bls\_period\_1st}}}{365.25}\right)^2 \right)^{1/3} \quad [\text{AU}]$$

##### Calculated Orbital Inclination ($c_{\text{planet\_i}}$)

To recover the orbital inclination, the code first transforms the semi-major axis into solar radii units ($a_{\text{R}_\odot} = c_{\text{planet\_a}} \times \frac{\text{AU}}{R_\odot}$), where $\frac{\text{AU}}{R_\odot} \approx 215.03$.

The mathematical transit chord execution term $\omega$ is calculated via:

$$\omega = \pi \left(\frac{f_{\text{best\_duration}}}{f_{\text{bls\_period\_1st}}}\right) \left(\frac{a_{\text{R}_\odot}}{f_{\text{radius}}}\right)$$

Using this chord term and the planet-to-star radius ratio $k = \sqrt{f_{\text{best\_depth}}}$, the script solves for the transit impact parameter $b$:

$$b = \sqrt{(1.0 + k)^2 - \omega^2}$$

Since the impact parameter is geometrically defined as $b = \frac{a \cos i}{R_\star}$, the true orbital inclination angle in degrees is recovered via:

$$\cos i = b \left(\frac{f_{\text{radius}}}{a_{\text{R}_\odot}}\right) \implies c_{\text{planet\_i}} = \arccos(\cos i) \times \frac{180^\circ}{\pi}$$

The script clips the cosine term between $-1.0$ and $1.0$ to prevent numerical errors near edge-on configurations.

---

### Preamble – Importing & Reading

In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina']

from IPython.display import clear_output

import json
import yaml

import copy
import os
import subprocess
import sys
import time

import re

from pathlib import Path

import tempfile
import h5py

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from scipy.stats import median_abs_deviation, skew, kurtosis

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py

from wotan import flatten
from astropy.timeseries import BoxLeastSquares

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['olivedrab', 'steelblue', 'firebrick', 'goldenrod'])
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (0, 0)
plt.rcParams['figure.figsize'] = [6,4]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

### Auxiliary Functions

In [2]:
# --------------------------------------------------------------- #
# Detrending (Wotan Biweight + Sparse Grid + Failure Protections) #
# --------------------------------------------------------------- #

def detrend(time, flux, flag):

    # --- guard against empty arrays --- #
    
    if len(time) == 0 or len(flux) == 0 or len(flag) == 0:
        return np.array([]), np.array([])

    # --- convert normalized baseline --- #
    
    flux_norm = 1.0 + (flux / 1e6)

    # --- find contiguous blocks using array transitions --- #
    
    good_mask = (flag == 0.0)
    transitions = np.where(np.diff(good_mask.astype(int)) != 0)[0] + 1
    split_indices = np.concatenate(([0], transitions, [len(time)]))

    WINDOW = 3.0
    EDGE = 1/24
    STRIDE = 20

    detrend_chunks = []
    trend_chunks = []

    # --- loop over actual blocks --- #
    
    for lo, hi in zip(split_indices[:-1], split_indices[1:]):
        
        # --- guard against zero width slices --- #
        
        if lo == hi:
            continue
            
        # --- prevent running for flagged blocks --- #
        
        if not good_mask[lo]:
            detrend_chunks.append(np.array([np.nan] * (hi - lo)))
            trend_chunks.append(np.array([np.nan] * (hi - lo)))
            continue
            
        t_chunk = time[lo:hi]
        f_chunk = flux_norm[lo:hi]
        
        # --- additional validation checking degraded chunk structures --- #
        
        if len(t_chunk) == 0:
            continue
            
        chunk_duration = t_chunk[-1] - t_chunk[0]
        
        # --- dynamic parameter protection --- #

        current_edge = EDGE if chunk_duration > (2 * EDGE) else 0.0
        current_window = WINDOW if chunk_duration > WINDOW else (chunk_duration / 2.0)
        if current_window < 0.1: 
            current_window = 0.1
        
        try:

            # --- stride optimization --- #

            if len(t_chunk) > (STRIDE * 2):
                t_sparse = t_chunk[::STRIDE]
                f_sparse = f_chunk[::STRIDE]
                
                if (len(t_chunk) - 1) % STRIDE != 0:
                    t_sparse = np.append(t_sparse, t_chunk[-1])
                    f_sparse = np.append(f_sparse, f_chunk[-1])
                    
                _, tmp_tr_sparse = flatten(
                    time=t_sparse,
                    flux=f_sparse,
                    method='biweight',
                    window_length=current_window,
                    edge_cutoff=current_edge,
                    return_trend=True
                )
                
                good_trend_mask = np.isfinite(tmp_tr_sparse)
                
                if np.any(good_trend_mask):
                    valid_times = t_sparse[good_trend_mask]
                    t_start, t_end = valid_times[0], valid_times[-1]
                    
                    tmp_tr = np.interp(t_chunk, t_sparse[good_trend_mask], tmp_tr_sparse[good_trend_mask])
                    tmp_tr[(t_chunk < t_start) | (t_chunk > t_end)] = np.nan
                else:
                    tmp_tr = np.full_like(t_chunk, np.nan)
                    
                tmp_detr = f_chunk / tmp_tr

            else:
                
                # --- fallback for very short blocks --- #
                
                tmp_detr, tmp_tr = flatten(
                    time=t_chunk,
                    flux=f_chunk,
                    method='biweight',
                    window_length=current_window,
                    edge_cutoff=current_edge,
                    return_trend=True
                )
        except Exception:
            
            # --- protect execution flow by filling array --- #
            
            tmp_tr = np.full_like(t_chunk, np.nan)
            tmp_detr = np.full_like(t_chunk, np.nan)
        
        detrend_chunks.append(tmp_detr)
        trend_chunks.append(tmp_tr)

    # --- reassemble and return results --- #
    
    full_detrend = np.concatenate(detrend_chunks) if detrend_chunks else np.array([])
    full_trend = np.concatenate(trend_chunks) if trend_chunks else np.array([])
    
    return full_trend, full_detrend

# --------------------------------------------------------- #
# Detection (Astropy Box Least Squares + Crash Protections) #
# --------------------------------------------------------- #

import numpy as np
from astropy.timeseries import BoxLeastSquares

def detect(time, detrend_data):
    if len(time) == 0 or len(detrend_data) == 0:
        return None, None, np.array([])

    # --- strip invalid entries --- #
    
    clean_mask = np.isfinite(time) & np.isfinite(detrend_data)
    if np.sum(clean_mask) < 2:
        return None, None, np.array([])

    tls_time_raw = time[clean_mask]
    tls_flux_raw = detrend_data[clean_mask]

    # --- guard against invalid time range sequencing --- #
    
    if tls_time_raw[-1] <= tls_time_raw[0]:
        return None, None, np.array([])

    # --- fast data binning --- #
    
    bin_size = 0.25 / 24.0 
    bin_edges = np.arange(tls_time_raw[0], tls_time_raw[-1] + bin_size, bin_size)

    if len(bin_edges) < 2:
        return None, None, np.array([])

    bin_counts, _ = np.histogram(tls_time_raw, bins=bin_edges)
    bin_flux, _ = np.histogram(tls_time_raw, bins=bin_edges, weights=tls_flux_raw)

    valid_bins = bin_counts > 0
    if not np.any(valid_bins):
        return None, None, np.array([])

    binned_time = (bin_edges[:-1] + bin_size / 2.0)[valid_bins]
    binned_flux = bin_flux[valid_bins] / bin_counts[valid_bins]

    # --- explicit period array --- #
    
    explicit_periods = np.linspace(35.0, 700.0, 10000)
    trial_durations = np.linspace(3.0 / 24.0, 36.0 / 24.0, 5)

    try:
        
        # --- execute transit search within safety wrapper --- #
        
        model = BoxLeastSquares(binned_time, binned_flux)
        results = model.power(
            period=explicit_periods, 
            duration=trial_durations,
            oversample=1
        )

        try:
            peak_indices, _ = find_peaks(results.power, distance=50)
            peak_indices = peak_indices[np.argsort(results.power[peak_indices])][-10:]
        except Exception:
            peak_indices = np.array([])

        return model, results, peak_indices
        
    except Exception:
        return None, None, np.array([])

### Features Generation

In [3]:
teff_sun = 5.778e3
m_sun = 1.98919e33
r_sun = 6.9599e10
lum_sun = 3.828e33
logg_sun = 4.438e0
rot_sun = 2.6e1
sig_sun = 3e1

r_earth = 9.17e-2
p_earth = 3.65256e2 
teq_earth = 2.55e2

def save(file_path, group_name, group, time, detrend_data, bls_model, bls_results, peak_indices, out_t, out_f):
    
    # --- initialize quality tracker --- #
    
    f_goodness_flag = True
    
    # --------------------------- #
    # Establish Ground Truth (t_) #
    # --------------------------- #

    raw_yaml = group.attrs['config_used']
    meta_dat = yaml.safe_load(raw_yaml)

    t_act_sigma = meta_dat['Activity']['Sigma']
    t_act_tau = meta_dat['Activity']['Tau']

    t_act_spot = meta_dat['Activity']['Spot']['Enable']

    t_act_flare = meta_dat['Activity']['Flare']['Enable']
    t_act_flare_per = meta_dat['Activity']['Flare']['MeanPeriod']
    t_act_flare_amp = meta_dat['Activity']['Flare']['Amplitude']
    t_act_flare_dur = meta_dat['Activity']['Flare']['MeanDuration']

    t_planet = meta_dat['Transit']['Enable']

    t_planet_p = meta_dat['Transit']['OrbitalPeriod']
    t_planet_r = meta_dat['Transit']['PlanetRadius'] / r_earth
    t_planet_a = meta_dat['Transit']['PlanetSemiMajorAxis']
    t_planet_i = meta_dat['Transit']['OrbitalAngle']

    # ----------------------------- #
    # Establish Feature Values (f_) #
    # ----------------------------- #
    
    raw = group['txt']
    text = raw.asstr()[()]

    target = ''
    for line in text.splitlines():
        if 'mass =' in line:
            target = line
            break
    
    matches = re.findall(r'(\w+)\s*=\s*([\d.]+)', target)
    params = {key: float(val) for key, val in matches}
    
    f_mass = params['mass']
    f_teff = params['teff']
    f_radius = params['radius']
    f_logg = params['logg']
    f_lum = params['lum']

    if f_radius > 0:
        f_dens = f_mass / f_radius**3
    else:
        f_dens = np.nan
        f_goodness_flag = False

    for line in text.splitlines():
        if 'magnitude =' in line:
            target = line
            break
    
    matches = re.findall(r'(\w+)\s*=\s*([\d.]+)', target)
    params = {key: float(val) for key, val in matches}
    
    f_mag = params['magnitude']

    # ------------------------------- #
    # Establish Predicted Values (p_) #
    # ------------------------------- #

    # --- compute stellar activity hostility --- #
    
    k_sigma = 0.0010
    k_flare = 0.0025
    
    if t_act_tau > 0:
        tau_term = t_act_sigma / t_act_tau
    else:
        tau_term = np.nan
        f_goodness_flag = False

    if t_act_flare_per > 0:
        flare_term = (t_act_flare_amp * t_act_flare_dur) / t_act_flare_per
    else:
        flare_term = np.nan
        f_goodness_flag = False

    p_B = 1.0 - np.exp(-k_sigma * tau_term) if np.isfinite(tau_term) else np.nan
    p_F = 1.0 - np.exp(-k_flare * t_act_flare * flare_term) if np.isfinite(flare_term) else np.nan
    p_H = 1.0 - (1.0 - p_B) * (1.0 - p_F) if (np.isfinite(p_B) and np.isfinite(p_F)) else np.nan

    # --- compute radial constraint --- #
    
    p_R = np.exp(-15 * (t_planet_r - 1.0)**4)

    # --- compute habitable zone temperature score --- #
    
    if t_planet_a > 0:
        p_teq = f_teff * (f_radius / (430 * t_planet_a))**(1/2) * (1.0 - 0.3)**(1/4)
        p_S = (f_radius / t_planet_a)**2 * (f_teff / teff_sun)**4
    else:
        p_teq = np.nan
        p_S = np.nan
        f_goodness_flag = False

    ts = f_teff - teff_sun
    HZ_inner = 1.1074e0 + 1.3323e-4 * ts + 1.5804e-8 * ts**2 - 8.3081e-12 * ts**3 - 1.9314e-15 * ts**4
    HZ_outer = 0.3560 + 6.1713e-5 * ts + 1.6981e-9 * ts**2 - 3.1982e-12 * ts**3 - 5.5753e-16 * ts**4

    Smid = (HZ_inner + HZ_outer) / 2.0
    Shalf = (HZ_inner - HZ_outer) / 2.0

    if Shalf > 0 and np.isfinite(p_S):
        p_xeff = (p_S - Smid) / Shalf
        p_T = np.exp(-p_xeff**6)
    else:
        p_xeff = np.nan
        p_T = np.nan
        f_goodness_flag = False

    # --- compute overall habitability score --- #
    
    p_C = (0.5 * p_T + 0.5 * p_R) * (1.0 - p_H) if (np.isfinite(p_T) and np.isfinite(p_H)) else np.nan

    # --- prepare clean arrays --- ##
    
    clean_mask = np.isfinite(time) & np.isfinite(detrend_data)
    time_clean = time[clean_mask]
    data_clean = detrend_data[clean_mask]

    # --- lightcurve variability and noise statistics --- #
    
    if len(data_clean) >= 5:
        f_lc_rms = float(np.std(data_clean))
        f_lc_mad = float(median_abs_deviation(data_clean, scale='normal'))
        f_lc_lo_1perc = float(np.percentile(data_clean, 1))
        f_lc_lo_5perc = float(np.percentile(data_clean, 5))
        f_lc_hi_1perc = float(np.percentile(data_clean, 99))
        f_lc_hi_5perc = float(np.percentile(data_clean, 95))
        f_lc_skew = float(skew(data_clean))
        f_lc_kurt = float(kurtosis(data_clean))
    else:
        f_goodness_flag = False
        f_lc_rms = np.nan
        f_lc_mad = np.nan
        f_lc_lo_1perc = np.nan
        f_lc_lo_5perc = np.nan
        f_lc_hi_1perc = np.nan
        f_lc_hi_5perc = np.nan
        f_lc_skew = np.nan
        f_lc_kurt = np.nan

    # --- block least squares analysis --- #
    
    if bls_results is not None and bls_model is not None and len(bls_results.power) > 0:
        power = bls_results.power
        periods = bls_results.period
        
        max_idx = np.argmax(power)
        f_bls_period_1st = float(periods[max_idx])
        f_bls_power_1st = float(power[max_idx])
        
        if peak_indices is not None and len(peak_indices) > 0:
            peak_powers = power[peak_indices]
            sorted_peak_args = np.argsort(peak_powers)[::-1]
            top_indices = peak_indices[sorted_peak_args]
            
            if len(top_indices) > 1 and top_indices[1] != max_idx:
                f_bls_period_2nd = float(periods[top_indices[1]])
                f_bls_power_2nd = float(power[top_indices[1]])
            else:
                f_bls_period_2nd, f_bls_power_2nd = np.nan, np.nan
                
            if len(top_indices) > 2 and top_indices[2] != max_idx:
                f_bls_period_3rd = float(periods[top_indices[2]])
                f_bls_power_3rd = float(power[top_indices[2]])
            else:
                f_bls_period_3rd, f_bls_power_3rd = np.nan, np.nan
        else:
            f_bls_period_2nd, f_bls_power_2nd = np.nan, np.nan
            f_bls_period_3rd, f_bls_power_3rd = np.nan, np.nan
            
        f_best_initial = bls_results.transit_time[max_idx]
        f_best_duration = bls_results.duration[max_idx]
        f_best_depth = bls_results.depth[max_idx]
        
        if f_bls_period_1st > 0:
            f_period_ratio_21 = f_bls_period_2nd / f_bls_period_1st
            f_period_ratio_31 = f_bls_period_3rd / f_bls_period_1st
            f_dur_per_ratio = f_best_duration / f_bls_period_1st
        else:
            f_goodness_flag = False
            f_period_ratio_21 = np.inf if f_bls_period_2nd > 0 else np.nan
            f_period_ratio_31 = np.inf if f_bls_period_3rd > 0 else np.nan
            f_dur_per_ratio = np.inf if f_best_duration > 0 else np.nan

        # --- primary transit profile geometry and consistency --- #

        if f_bls_period_1st > 0 and f_best_duration > 0 and len(data_clean) > 0:
            stats = bls_model.compute_stats(f_bls_period_1st, f_best_duration, f_best_initial)
            depth_val, depth_err = stats['depth']
            
            f_bls_snr = float(depth_val / depth_err) if depth_err > 0 else np.inf
            f_bls_num_actual = int(np.sum(stats['per_transit_count'] > 0))
            f_bls_num_expect = len(stats['transit_times'])
            
            if f_bls_num_expect > 0:
                f_bls_num_ratio = float(f_bls_num_actual / f_bls_num_expect)
            else:
                f_goodness_flag = False
                f_bls_num_ratio = np.nan
        else:
            f_goodness_flag = False
            f_bls_snr = np.nan
            f_bls_num_actual = np.nan
            f_bls_num_expect = np.nan
            f_bls_num_ratio = np.nan

        # --- calculate shape factor --- #

        if f_bls_period_1st > 0 and len(data_clean) >= 5:
            fold_phase = ((time_clean - f_best_initial) % f_bls_period_1st) / f_bls_period_1st
            fold_phase[fold_phase > 0.5] -= 1.0
            
            half_duration_phase = f_best_duration / (2.0 * f_bls_period_1st)
            in_transit = np.abs(fold_phase) < half_duration_phase
            transit_flux = data_clean[in_transit]

            if len(transit_flux) >= 5:
                depths = 1.0 - transit_flux
                max_d = np.max(depths)
                mean_d = np.mean(depths)
                f_bls_trans_shape_fac = float(mean_d / max_d) if max_d > 0 else np.nan
                if max_d <= 0:
                    f_goodness_flag = False
            else:
                f_goodness_flag = False
                f_bls_trans_shape_fac = np.nan
        else:
            f_goodness_flag = False
            transit_flux = np.array([])
            f_bls_trans_shape_fac = np.nan

        # --- points in transit ratio --- #

        expected_points_in_transit = len(data_clean) * f_dur_per_ratio
        if expected_points_in_transit > 0:
            f_points_in_transit_ratio = float(len(transit_flux) / expected_points_in_transit)
        else:
            f_goodness_flag = False
            f_points_in_transit_ratio = np.nan

        # --- density ratio verification --- #

        if f_best_duration > 0 and f_bls_period_1st > 0 and np.isfinite(f_dens) and f_dens > 0:
            G_cgs = 6.6743e-8
            sun_dens_cgs = 1.408 
            
            P_sec = f_bls_period_1st * 86400.0
            T_sec = f_best_duration * 86400.0
            
            rho_calc_cgs = (3.0 * np.pi / G_cgs) * ((P_sec) / (np.pi * T_sec))**3
            rho_star_cgs = f_dens * sun_dens_cgs
            f_density_ratio = float(rho_calc_cgs / rho_star_cgs)
        else:
            f_goodness_flag = False
            f_density_ratio = np.nan

        # -------------------------------- #
        # Establish Calculated Values (c_) #
        # -------------------------------- #

        if f_best_depth >= 0 and f_radius > 0:
            c_planet_r = f_radius * np.sqrt(f_best_depth) * 109.12
        else:
            c_planet_r = np.nan
            f_goodness_flag = False

        if f_bls_period_1st > 0 and f_mass > 0:
            c_planet_a = (f_mass * (f_bls_period_1st / 365.25)**2)**(1/3)
        else:
            c_planet_a = np.nan
            f_goodness_flag = False

        if (f_bls_period_1st > 0 and f_best_duration > 0 and f_radius > 0 and 
                np.isfinite(c_planet_a) and c_planet_a > 0 and np.isfinite(c_planet_r)):
            
            au_to_rsun = 1.495978707e13 / r_sun
            a_rsun = c_planet_a * au_to_rsun
            k_ratio = np.sqrt(f_best_depth)
            
            omega_term = np.pi * (f_best_duration / f_bls_period_1st) * (a_rsun / f_radius)
            bb = (1.0 + k_ratio)**2 - omega_term**2            
            if bb >= 0:
                b_impact = np.sqrt(bb)
                cos_i = b_impact * (f_radius / a_rsun)
                cos_i = np.clip(cos_i, -1.0, 1.0)
                c_planet_i = float(np.degrees(np.arccos(cos_i)))
            else:
                c_planet_i = np.nan
                f_goodness_flag = False
        else:
            c_planet_i = np.nan
            f_goodness_flag = False

    else:
        f_goodness_flag = False
        
        f_bls_period_1st, f_bls_power_1st = np.nan, np.nan
        f_bls_period_2nd, f_bls_power_2nd = np.nan, np.nan
        f_bls_period_3rd, f_bls_power_3rd = np.nan, np.nan
        
        f_best_initial, f_best_duration, f_best_depth = np.nan, np.nan, np.nan
        
        f_period_ratio_21 = np.nan
        f_period_ratio_31 = np.nan
        f_dur_per_ratio = np.nan
        
        f_bls_snr = np.nan
        f_bls_num_actual = np.nan
        f_bls_num_expect = np.nan
        f_bls_num_ratio = np.nan
        
        f_bls_trans_shape_fac = np.nan
        f_points_in_transit_ratio = np.nan
        f_density_ratio = np.nan
        
        c_planet_r = np.nan
        c_planet_a = np.nan
        c_planet_i = np.nan

    # --- derived calculations --- #
    
    c_R = np.exp(-15 * (c_planet_r - 1.0)**4)
    
    if c_planet_a > 0:
        c_teq = f_teff * (f_radius / (430 * c_planet_a))**(1/2) * (1.0 - 0.3)**(1/4)
        c_S = (f_radius / c_planet_a)**2 * (f_teff / teff_sun)**4
    else:
        c_teq = np.nan
        c_S = np.nan
        f_goodness_flag = False

    if Shalf > 0 and np.isfinite(c_S):
        c_xeff = (c_S - Smid) / Shalf
        c_T = np.exp(-c_xeff**6)
    else:
        c_xeff = np.nan
        c_T = np.nan
        f_goodness_flag = False
        
    # ------------ #
    # Saving Logic #
    # ------------ #

    id_dict = {'path': file_path, 'sim': group_name}

    local_vars = locals()

    t_dict = {k: local_vars[k] for k in sorted(local_vars.keys()) if k.startswith('t_')}
    p_dict = {k: local_vars[k] for k in sorted(local_vars.keys()) if k.startswith('p_')}
    truth_row = {**id_dict, **t_dict, **p_dict}
    df_truth_new = pd.DataFrame([truth_row])

    f_dict = {k: local_vars[k] for k in sorted(local_vars.keys()) if k.startswith('f_')}
    c_dict = {k: local_vars[k] for k in sorted(local_vars.keys()) if k.startswith('c_')}
    features_row = {**id_dict, **f_dict, **c_dict}
    df_features_new = pd.DataFrame([features_row])

    truth_file_exists = os.path.exists(out_t)
    df_truth_new.to_csv(
        out_t, 
        mode='a', 
        header=not truth_file_exists, 
        index=False
    )

    features_file_exists = os.path.exists(out_f)
    df_features_new.to_csv(
        out_f, 
        mode='a', 
        header=not features_file_exists, 
        index=False
    )

### Simulation Loop

In [4]:
# -------------------------------------------------------------------------
# FUNCTION 3: Main Processing Pipeline (Saving Removed)
# -------------------------------------------------------------------------
def process(input_names, output_truth = 'truth', output_features = 'features', append = False):
    """
    Iterates through a list of HDF5 input files, applies detrending and BLS detection 
    to each group. Output file parameter is kept in signature to ensure cell execution 
    compatibility but all file generation and storage logic has been removed.
    """
    start_batch = time.time()

    if not append and (os.path.exists(f'data/{output_truth}.csv') or os.path.exists(f'data/{output_features}.csv')):
        print('Files already exist')
        return
    
    # Pre-scan inputs to count total subgroups across all files for the status bar
    total_sims = 0
    file_mapping = {}
    for name in input_names:
        f_name = f'PSLS/data/{name}.hdf5'
        if not os.path.exists(f_name):
            continue
        with h5py.File(f_name, 'r') as f:
            keys = list(f.keys())
            file_mapping[f_name] = keys
            total_sims += len(keys)

    if total_sims == 0:
        print("No valid groups found in the provided files.")
        return
    
    # Print Initial Status
    empty_bar = '░' * 50
    initial_status = f'│ progress |{empty_bar}| 0% [running 1/{total_sims}] (00:00:00/--:--:--)          '
    print(f'{initial_status}')

    # Process files
    i = 0
    for f_name, group_keys in file_mapping.items():
        with h5py.File(f_name, 'r') as in_f:
            for group_name in group_keys:
                
                # Update Visual Status Bar
                if i > 0:
                    perc = (i / total_sims) * 100
                    bar = '█' * int(perc // 2) + '░' * (50 - int(perc // 2))
                    
                    current_run = time.time()
                    elapsed_batch = current_run - start_batch
                    average_time = elapsed_batch / i
                    estimated_batch = elapsed_batch + (total_sims - i) * average_time
                    
                    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed_batch))
                    estimated_str = time.strftime('%H:%M:%S', time.gmtime(estimated_batch))
                    
                    status = f'│ progress |{bar}| {perc:.0f}% [running {i + 1}/{total_sims}] ({elapsed_str}/{estimated_str})          '
                    
                    clear_output(wait=True)
                    print(f'{status}')
                
                sys.stdout.write(f'└── processing run {int(group_name[-5:])}          ')
                sys.stdout.flush()
                
                # 1. Extract Data with an explicit group safety block
                try:
                    flag = in_f[f'{group_name}/dat/flag'][:]
                    flux = in_f[f'{group_name}/dat/flux_var_ppm'][:]
                    time_data = in_f[f'{group_name}/dat/time_s'][:] / (60 * 60 * 24)
                except Exception:
                    # If internal group entries are unreadable, increment count and skip gracefully
                    i += 1
                    continue

                sys.stdout.write(f'\r└─┬ processing run {int(group_name[-5:])}          \n')
                sys.stdout.write(f'  ├── sparse wotan biweight detrending...          ')
                sys.stdout.flush()

                # 2. Detrend
                trend_arr, detrend_arr = detrend(time_data, flux, flag)

                sys.stdout.write(f'\r  ├── sparse wotan biweight detrending [done]          \n')
                sys.stdout.write(f'  └── astropy box least squares search...          ')
                sys.stdout.flush()

                # 3. Detect
                bls_model, bls_results, peak_indices = detect(time_data, detrend_arr)
                
                sys.stdout.write(f'\r  └── astropy box least squares search [done]          \n')
                sys.stdout.flush()

                save(f_name,
                     group_name,
                     in_f[group_name],
                     time_data,
                     detrend_arr,
                     bls_model,
                     bls_results,
                     peak_indices,
                     f'data/{output_truth}.csv',
                     f'data/{output_features}.csv')

                i += 1

    # Print Final Status
    elapsed_batch = time.time() - start_batch
    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed_batch))
    full_bar = '█' * 50
    final_status = f'│ progress |{full_bar}| 100% [finished {total_sims}/{total_sims}] ({elapsed_str}/{elapsed_str})          '
    
    clear_output(wait=True)
    print(f'{final_status}')

In [5]:
datasets = ['trainPlanet', 'trainStar', 'realPlanet', 'realStar']
Path('data/exampleTruth.csv').unlink(missing_ok=True)
Path('data/exampleFeatures.csv').unlink(missing_ok=True)
datasets = ['example']
process(datasets, output_truth='exampleTruth', output_features='exampleFeatures')

│ progress |██████████████████████████████████████████████████| 100% [finished 10/10] (00:00:41/00:00:41)          
